In [12]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import combinations
from rapidfuzz import fuzz

In [13]:
seed = 99
filepath_syn = Path(f"../data/synthetic/{seed}")
filepath_gt = Path(f"../data/ground_truth/{seed}")

customers_raw = pd.read_csv(filepath_syn/"customers_raw.csv")
products_raw = pd.read_csv(filepath_syn/"products_raw.csv")
orders_raw = pd.read_csv(filepath_syn/"orders_raw.csv")

customers_clean = pd.read_csv(filepath_gt/"customers_clean.csv")
products_clean = pd.read_csv(filepath_gt/"products_clean.csv")

In [14]:
def show_all(df, name=""):
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None,
                           "display.width", None):
        if name:
            print(f"=== {name}: {len(df)} rows ===")
        display(df)


def inspect_duplicates(df, subset=None, name=""):
    dups = df[df.duplicated(subset=subset, keep=False)]
    sort_cols = subset if subset else list(df.columns)
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None):
        print(f"=== {name}: {len(dups)} duplicate rows "
              f"(subset={subset or 'all columns'}) ===")
        display(dups.sort_values(by=sort_cols))

def inspect_fuzzy_duplicates(df, cols=None, exclude=None, threshold=85,
                             scorer=fuzz.ratio, name=""):
    exclude = set(exclude or [])
    if cols is None:                      
        cols = [c for c in df.columns if c not in exclude]
    sub = df[cols].fillna("").astype(str)
    recs = sub.to_dict("records")
    idx = df.index.tolist()

    rows = []
    for a, b in combinations(range(len(df)), 2):
        scores = {c: scorer(recs[a][c], recs[b][c]) for c in cols}
        avg = sum(scores.values()) / len(cols)
        if avg >= threshold:
            rows.append({"score": round(avg, 1), "idx_a": idx[a], "idx_b": idx[b],
                         **{f"{c}_a": recs[a][c] for c in cols},
                         **{f"{c}_b": recs[b][c] for c in cols}})

    result = (pd.DataFrame(rows).sort_values("score", ascending=False)
              .reset_index(drop=True))
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print(f"=== {name}: {len(result)} fuzzy pairs "
              f"(threshold={threshold}, cols={cols}) ===")
        display(result)
    return result

In [18]:
# Customers

show_all(customers_raw.sort_values("customer_id"), "customers_raw")             #sorted raw
show_all(customers_clean.sort_values("customer_id"), "customers_clean")         #sorted clean
#show_all(customers_raw, "customers_raw")                                        #unsorted
#show_all(customers_clean, "customers_clean")                                    #unsorted

# Duplicates

#inspect_duplicates(customers_raw, name="customers_raw")
inspect_duplicates(customers_clean, name="customers_clean")    # 0 expected

inspect_fuzzy_duplicates(customers_raw, cols=["full_name", "email"], threshold=80, name="customers_raw");

=== customers_raw: 540 rows ===


,customer_id,full_name,email,country,registered_at
151,1,Hanife Gute-Scheuermann,hermann-josefhering@example.org,Niederlande,September 25 2020
281,2,Monika Kraus,scholzadelheid@example.org,Netherlands,1690934400
250,3,Robby Zirme B.Eng.,nstoll@example.org,France,1701648000
374,4,Prof. Gunther Carsten,moechlichenmaik@example.com,GB,2020-05-12
375,5,Kristiane Albers,xhaase@example.net,CH,1599523200
488,6,Karen Gumprich MBA.,klemtflorian@example.org,FRA,2021-05-14
104,7,Lilija Gierschner B.A.,doerschneranatolij@example.org,FR,January 11 2024
411,8,Frau Valeska Löffler,fechnerlorenzo@example.net,schweiz,2020-01-18
402,9,Genoveva Ernst-Preiß,ygute@example.org,AT,June 12 2025
131,10,Heinz-Willi Hertrampf MBA.,nikolaustaesche@example.com,Italy,2020-06-22


=== customers_clean: 500 rows ===


,customer_id,full_name,email,country_code,registered_at
0,1,Hanife Gute-Scheuermann,hermann-josefhering@example.org,NL,2020-09-25
1,2,Monika Kraus,scholzadelheid@example.org,NL,2023-08-02
2,3,Robby Zirme B.Eng.,nstoll@example.org,FR,2023-12-04
3,4,Prof. Gunther Carsten,moechlichenmaik@example.com,GB,2020-05-12
4,5,Kristiane Albers,xhaase@example.net,CH,2020-09-08
5,6,Karen Gumprich MBA.,klemtflorian@example.org,FR,2021-05-14
6,7,Lilija Gierschner B.A.,doerschneranatolij@example.org,FR,2024-01-11
7,8,Frau Valeska Löffler,fechnerlorenzo@example.net,CH,2020-01-18
8,9,Genoveva Ernst-Preiß,ygute@example.org,AT,2025-06-12
9,10,Heinz-Willi Hertrampf MBA.,nikolaustaesche@example.com,IT,2020-06-22


=== customers_clean: 0 duplicate rows (subset=all columns) ===


,customer_id,full_name,email,country_code,registered_at


=== customers_raw: 33 fuzzy pairs (threshold=80, cols=['full_name', 'email']) ===


,score,idx_a,idx_b,full_name_a,email_a,full_name_b,email_b
0,100.0,25,300,Marieluise Schmidtke,tmargraf@example.org,Marieluise Schmidtke,tmargraf@example.org
1,100.0,372,464,Vittorio Beer,jovanjunk@example.net,Vittorio Beer,jovanjunk@example.net
2,100.0,43,296,Klaus Dieter Beckmann,lindaukrzysztof@example.com,Klaus Dieter Beckmann,lindaukrzysztof@example.com
3,100.0,61,495,Annedore Hesse,ifaust@example.net,Annedore Hesse,ifaust@example.net
4,100.0,96,261,Univ.Prof. Ivanka Fröhlich,brit28@example.net,Univ.Prof. Ivanka Fröhlich,brit28@example.net
5,100.0,91,529,Joachim Huhn,ldavids@example.net,Joachim Huhn,ldavids@example.net
6,100.0,75,454,Sieglinde Gutknecht MBA.,esiering@example.org,Sieglinde Gutknecht MBA.,esiering@example.org
7,100.0,69,135,Giorgio Rogge,anna-maria54@example.com,Giorgio Rogge,anna-maria54@example.com
8,100.0,64,143,Gerdi Striebitz-Kühnert,ngnatz@example.net,Gerdi Striebitz-Kühnert,ngnatz@example.net
9,100.0,93,136,Dr. Linda Hendriks,sigurd51@example.net,Dr. Linda Hendriks,sigurd51@example.net


In [16]:
# Products

show_all(products_raw.sort_values("product_id"), "products_raw")     #sorted
show_all(products_clean.sort_values("product_id"), "products_clean") 
#show_all(products, "products")                               #unsorted

# Duplicates

inspect_duplicates(products_raw, name="products_raw")
#inspect_duplicates(products_clean, subset=["product_id"], name="products_clean")

=== products_raw: 100 rows ===


,product_id,name,category,price_eur,in_stock
0,1,Multi-layered systemic attitude,Sports,30.78 EUR,no
1,2,Synergistic encompassing array,Electronics,336.58,yes
2,3,Expanded zero-defect parallelism,Sports,"105,09",1
3,4,Organized background migration,Home,436.74 EUR,1
4,5,Devolved discrete ability,Sports,"123,92",yes
5,6,Versatile actuating circuit,Electronics,436.44,1
6,7,Focused hybrid archive,Toys,59.22,1
7,8,Decentralized homogeneous superstructure,Books,301.34,yes
8,9,Cloned heuristic portal,Clothing,475.06 EUR,true
9,10,Pre-emptive national hardware,Electronics,358.81 EUR,yes


=== products_clean: 100 rows ===


,product_id,name,category,price_eur,in_stock
0,1,Multi-layered systemic attitude,Sports,30.78,False
1,2,Synergistic encompassing array,Electronics,336.58,True
2,3,Expanded zero-defect parallelism,Sports,105.09,True
3,4,Organized background migration,Home,436.74,True
4,5,Devolved discrete ability,Sports,123.92,True
5,6,Versatile actuating circuit,Electronics,436.44,True
6,7,Focused hybrid archive,Toys,59.22,True
7,8,Decentralized homogeneous superstructure,Books,301.34,True
8,9,Cloned heuristic portal,Clothing,475.06,True
9,10,Pre-emptive national hardware,Electronics,358.81,True


=== products_raw: 0 duplicate rows (subset=all columns) ===


,product_id,name,category,price_eur,in_stock


In [17]:
# Orders

show_all(orders.sort_values("order_id"), "orders")          #sorted
#show_all(orders, "orders")                                 #unsorted

# Duplicates

inspect_duplicates(orders, name="orders")
#inspect_duplicates(orders, subset=["order_id"], name="orders")

NameError: name 'orders' is not defined